# 第 6 周 · 周二：用户画像 SQL 实战——用SQL构建用户标签体系

> 难度：⭐⭐⭐ | 定位：掌握用SQL构建用户画像标签表的方法论，理解多维度聚合的工程模式
> 前置依赖：W5窗口函数/W6D1复杂SQL拆解 | 后续用途：W7-W8 AB实验用户分群、精细化运营
> 面试地位：大厂数据分析师日常工作核心技能，标签体系建设必问

---

## 📘 区块1：核心概念与第一性原理

### 1.1 用户画像在SQL中的本质

用户画像在工程实现上就是一个聚合操作：将用户的多行行为数据，聚合成一行用户级别的特征记录。

```
原始数据(多行/用户)：          画像表(一行/用户)：
user_id  order_date  amount    user_id  total_spend  active_days  fav_category  value_tier
1        2026-01-01  200       1        800          15           数码          高价值
1        2026-01-05  350  →    2        450          8            食品          中等
1        2026-01-12  250       3        1200         22           服装          高价值
2        2026-01-03  150
2        2026-01-10  300
```

这就是用户画像的工程本质：**GROUP BY user_id + 一堆CASE WHEN和聚合函数**。

### 1.2 从第一性原理推导：为什么画像需要"一行一个用户"？

用户画像是用来做用户分群和精准营销的。如果你要筛选"高价值且偏好数码的用户"，SQL的逻辑必须是：
```sql
SELECT user_id FROM user_profile
WHERE value_tier = '高价值' AND fav_category = '数码';
```

如果画像不是user_id粒度(比如一个用户有多行)，这个简单的筛选就做不到。**一行一个用户是用户画像最基本也最重要的约束。**

### 1.3 画像标签的三个层次

| 层次 | 说明 | 示例 | 更新频率 | 实现方式 |
|------|------|------|----------|---------|
| 基础属性 | 注册信息、渠道来源 | 年龄、性别、注册渠道、城市 | 几乎不变 | 直接从用户表取 |
| 行为统计 | 从行为数据聚合 | 累计消费、购买频次、活跃天数 | 每日更新 | SQL聚合计算 |
| 模型预测 | 算法预测标签 | 流失概率、LTV预测、品类偏好 | 每周/月更新 | 机器学习模型 |

SQL主要覆盖前两个层次，第三个层次需要机器学习模型。但前两个层次已经能覆盖80%的日常运营需求。

### 1.4 画像标签的类型学

**数值型标签**：total_spend=1500, order_count=12, avg_order=125.0
- 可用于条件筛选(>1000)和分段
- 常用于统计分析和趋势监控

**分类型标签**：fav_category='数码', city_tier='一线', channel='organic'
- 用于分群和交叉分析
- 是精准营销的目标维度

**布尔型标签**：is_high_value=1, has_repurchased=1, is_churned=0
- 用于快速筛选和标识
- 方便下游系统直接过滤

**时间型标签**：last_order_date='2026-01-15', days_since_last=7
- 用于时效性判断
- 驱动自动化召回和激活策略

### 1.5 用户画像 vs CDP vs CRM

很多公司一上来就买CDP(Customer Data Platform)工具，花几十万建标签体系。但在数据团队能力足够的情况下，SQL就能完成80%的画像工作：
- CDP的优势：可视化标签配置、实时计算、标签市场
- SQL画像的优势：灵活、可控、没有黑盒、零额外成本
- 建议：先用SQL构建核心画像(20-30个标签)，验证业务价值后再考虑CDP

### 1.6 面试高频考点

**面试官常问**：
- "如何用SQL构建用户画像表？"
- "用户画像的标签有哪些类型？"
- "如何处理画像标签的更新？"
- "画像表的粒度为什么必须是user_id？"
- "RFM模型是什么？怎么用SQL实现？"

**回答要点**：
- GROUP BY user_id + CASE WHEN多维度聚合
- 数值型、分类型、布尔型、时间型四类
- 用分区表每日增量更新活跃标签
- 画像需要支持单用户筛选，粒度必须是user_id
- RFM = Recency(最近) + Frequency(频次) + Monetary(金额)

## 🧩 区块2：知识点详细拆解

### 2.1 价值标签：RFM模型在SQL中的实现

**为什么重要**：RFM(Recency最近购买时间、Frequency购买频次、Monetary消费金额)是最经典的用户价值评估模型。1974年提出，到现在仍然是最实用的用户分群工具。

**三个维度的业务含义**：
- R(Recency)：最近一次购买距今多少天。越近越好——说明用户还在活跃。
- F(Frequency)：购买了多少次。越多越好——说明用户习惯已经养成。
- M(Monetary)：花了多少钱。越多越好——说明用户消费能力强。

**SQL实现(RFM打分)**：
```sql
WITH rfm AS (
  SELECT user_id,
    DATEDIFF('2026-02-01', MAX(order_date)) AS recency,
    COUNT(DISTINCT order_id) AS frequency,
    SUM(amount) AS monetary
  FROM orders
  GROUP BY user_id
),
rfm_scores AS (
  SELECT user_id,
    CASE WHEN recency <= 7 THEN 3
         WHEN recency <= 30 THEN 2 ELSE 1 END AS r_score,
    CASE WHEN frequency >= 10 THEN 3
         WHEN frequency >= 5 THEN 2 ELSE 1 END AS f_score,
    CASE WHEN monetary >= 1000 THEN 3
         WHEN monetary >= 300 THEN 2 ELSE 1 END AS m_score
  FROM rfm
)
SELECT user_id, r_score, f_score, m_score,
  r_score + f_score + m_score AS rfm_total,
  CASE WHEN r_score + f_score + m_score >= 8 THEN '重要价值'
       WHEN r_score + f_score + m_score >= 5 THEN '一般价值'
       ELSE '低价值' END AS rfm_segment
FROM rfm_scores;
```

**RFM分段解读**：
| RFM总分 | 分段名称 | 用户特征 | 运营策略 |
|---------|---------|---------|--------|
| 8-9 | 重要价值 | 最近买过+常买+高消费 | VIP维护，升级服务 |
| 6-7 | 重要发展 | 最近活跃+消费尚可 | 提升客单价，交叉销售 |
| 4-5 | 重要保持 | 消费高但最近不活跃 | 召回优惠，重新激活 |
| 3 | 一般价值 | 各方面中等 | 常规运营，引导复购 |

### 2.2 活跃度标签：从最后一次行为推断

**为什么重要**：活跃度是时效性最强的标签，决定用户是否需要被"唤醒"。一个昨天还在买的低价值用户，比一个3月没来的高价值用户更值得运营关注。

**SQL实现**：
```sql
SELECT user_id,
  MAX(order_date) AS last_active_date,
  DATEDIFF(CURRENT_DATE(), MAX(order_date)) AS days_since_last,
  CASE WHEN DATEDIFF(CURRENT_DATE(), MAX(order_date)) <= 7 THEN '活跃'
       WHEN DATEDIFF(CURRENT_DATE(), MAX(order_date)) <= 30 THEN '沉默'
       WHEN DATEDIFF(CURRENT_DATE(), MAX(order_date)) <= 90 THEN '预流失'
       ELSE '流失' END AS active_status
FROM orders
GROUP BY user_id;
```

**活跃定义的业务差异**：
| 产品类型 | 活跃标准 | 沉默标准 | 原因 |
|---------|---------|---------|------|
| 电商(日用品) | 7天 | 30天 | 高频消费品 |
| 电商(大件) | 30天 | 90天 | 低频高客单价 |
| 社交App | 1天 | 7天 | 日活型产品 |
| 工具App | 7天 | 30天 | 按需使用 |
| SaaS/企业软件 | 30天 | 90天 | 按月使用 |

**关键注意**：新用户(注册<7天没有历史行为)的活跃状态需要特殊处理——不能用MAX(order_date)判断，应该从注册表获取。

### 2.3 品类偏好标签：找出用户最喜欢的品类

**为什么重要**：品类偏好是个性化推荐和精准营销的基础。知道用户爱买什么，才能推荐什么。

**方法1：按消费金额(主流做法)**
```sql
SELECT user_id, category AS fav_category
FROM (
  SELECT user_id, category,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY SUM(amount) DESC) AS rn
  FROM orders
  GROUP BY user_id, category
) t WHERE rn = 1;
```

**方法2：按购买频次**
```sql
SELECT user_id, category AS most_frequent_category
FROM (
  SELECT user_id, category,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC) AS rn
  FROM orders
  GROUP BY user_id, category
) t WHERE rn = 1;
```

**方法3：综合偏好(金额x0.6 + 频次x0.4)**
给金额和频次标准化后加权求和，避免只买过一次大件就被判定为"偏好"。

**选择建议**：
- 客单价差异大(有奢侈品有日用品) → 用频次
- 客单价相对均匀 → 用金额
- 需要综合判断 → 加权求和

### 2.4 消费行为标签：刻画消费模式

**为什么重要**：消费行为标签描述了用户"怎么买"，比"买了什么"更深入。两个消费金额相同的用户，消费行为可能完全不同。

**关键指标及SQL**：
```sql
SELECT user_id,
  ROUND(AVG(amount), 1) AS avg_order_value,          -- 客单价
  ROUND(SUM(CASE WHEN amount > 500 THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1) AS big_order_pct,        -- 大单占比
  COUNT(DISTINCT category) AS category_diversity,    -- 跨品类数
  ROUND(STDDEV(amount), 1) AS spending_volatility,   -- 消费波动
  MAX(amount) AS max_single_order,                   -- 最大单笔
  COUNT(DISTINCT DATE_FORMAT(order_date, '%Y-%m'))
    AS active_months                                  -- 活跃月数
FROM orders
GROUP BY user_id;
```

**标签解读**：
- 客单价高 + 大单占比高 = 精品型消费者，适合推高端商品
- 客单价低 + 跨品类多 = 探索型消费者，适合推新品尝鲜
- 消费波动大 = 冲动型消费者，适合限时折扣
- 活跃月数多 + 消费稳定 = 习惯型消费者，适合订阅制

### 2.5 渠道来源标签：分析获客质量

**为什么重要**：不同渠道来的用户质量差异可能很大。渠道标签帮助做获客ROI分析——知道哪个渠道的用户LTV最高，就在哪个渠道多投钱。

```sql
SELECT u.user_id, u.channel,
  COUNT(o.order_id) AS order_count,
  SUM(o.amount) AS total_spend,
  ROUND(SUM(o.amount) * 1.0 / COUNT(DISTINCT o.order_id), 1) AS avg_order_value,
  DATEDIFF(MAX(o.order_date), MIN(o.order_date)) AS lifecycle_days
FROM users u
LEFT JOIN orders o ON u.user_id = o.user_id
GROUP BY u.user_id, u.channel;
```

**渠道质量分析维度**：
- 各渠道用户的平均消费(LTV初估)
- 各渠道用户的7日/30日留存率
- 各渠道用户的平均购买周期
- 各渠道的高价值用户占比

### 2.6 生命周期标签：用户在哪个阶段？

**为什么重要**：不同生命周期阶段的用户需要不同的运营策略。给新用户发召回券是浪费，给流失用户发新手引导也是浪费。

**生命周期五阶段模型**：
| 阶段 | 定义 | 运营策略 | 关键指标 |
|------|------|---------|--------|
| 新用户 | 注册<7天 | 引导和教育，完成关键行为 | 激活率 |
| 成长期 | 7-30天，活跃高 | 促进复购和习惯养成 | 复购率 |
| 成熟期 | 30-90天，稳定购买 | 维持和VIP运营 | 留存率 |
| 衰退期 | 90-180天，活跃下降 | 召回和优惠激活 | 召回率 |
| 流失期 | >180天未活跃 | 大力度召回或放弃 | 复活率 |

```sql
SELECT user_id,
  DATEDIFF(CURRENT_DATE(), MIN(order_date)) AS days_since_first,
  DATEDIFF(CURRENT_DATE(), MAX(order_date)) AS days_since_last,
  CASE WHEN DATEDIFF(CURRENT_DATE(), MIN(order_date)) <= 7 THEN '新用户'
       WHEN DATEDIFF(CURRENT_DATE(), MAX(order_date)) <= 30 THEN '活跃用户'
       WHEN DATEDIFF(CURRENT_DATE(), MAX(order_date)) <= 90 THEN '沉默用户'
       ELSE '流失用户' END AS lifecycle_stage
FROM orders
GROUP BY user_id;
```

### 2.7 画像表的设计原则

**原则1：一行一用户(不可违背)**
如果违反，所有的用户筛选逻辑都会出错。这是画像表的"第一性原理"。

**原则2：标签互斥**
value_tier不能既是"高价值"又是"中等"——用CASE WHEN时注意条件的互斥性。条件从上到下匹配，第一个满足的就返回，所以高阈值条件要放在前面。

**原则3：可增量更新**
活跃标签需要每天更新，用分区表每天跑增量：
```sql
INSERT OVERWRITE TABLE user_profile PARTITION (dt = '2026-01-15')
SELECT user_id, ... FROM orders WHERE dt = '2026-01-15';
```

**原则4：区分"未知"和"没有"**
- 没有购买记录 != 消费金额为0(可能是数据缺失)
- 用NULL表示"未知"(数据不存在)，用0表示"没有"(数据存在但值为0)
- 用COALESCE处理NULL：`COALESCE(total_spend, -1)` 表示"未知"

**原则5：时效标签和统计标签分开管理**
- 时效标签(活跃状态、沉默天数)：每日更新
- 统计标签(品类偏好、消费波动)：每周更新
- 基础标签(注册渠道、城市)：几乎不更新

### 2.8 画像表的存储与性能

**为什么重要**：画像表会被高频查询(运营可能每天查询几十次)，性能直接影响运营效率。

**优化策略**：
- 按日期分区存储，每天一个分区
- 对user_id建立主键索引
- 对常用筛选字段(value_tier、active_status、fav_category)建立索引
- 宽表设计：把所有标签放在一张表里，避免多表JOIN
- 预聚合：常用的画像统计(各层用户数)可以预计算存储
- 冷热分离：最近30天画像放热存储(SSD)，历史画像放冷存储(HDD)

## 💻 区块3：动手练习

### 步骤0：数据准备

生成500条订单模拟数据(50个用户，4个品类，3个月时间跨度)

### 步骤1：基础练习——构建用户画像表

用GROUP BY user_id + CASE WHEN构建包含价值分层和活跃状态的基础画像表

In [ ]:
import sqlite3, pandas as pd, numpy as np
np.random.seed(42)
conn = sqlite3.connect(':memory:')

n=500
df=pd.DataFrame({
    'user_id':np.random.choice(range(1,51),n),
    'category':np.random.choice(['服装','数码','食品','美妆'],n),
    'amount':np.round(np.random.lognormal(4,0.8,n),2),
    'order_date':np.random.choice(pd.date_range('2025-11-01','2026-01-31'),n)
})
df['order_date']=df['order_date'].astype(str)
df.to_sql('orders',conn,index=False)

q="""SELECT user_id,
  SUM(amount) AS total_spend,
  COUNT(*) AS order_cnt,
  ROUND(AVG(amount),1) AS avg_order,
  (JULIANDAY('2026-02-01')-JULIANDAY(MAX(order_date))) AS days_since_last,
  CASE WHEN SUM(amount)>=1000 THEN '高价值'
       WHEN SUM(amount)>=300 THEN '中等' ELSE '低价值' END AS value_tier,
  CASE WHEN JULIANDAY('2026-02-01')-JULIANDAY(MAX(order_date))<=7 THEN '活跃'
       WHEN JULIANDAY('2026-02-01')-JULIANDAY(MAX(order_date))<=30 THEN '沉默'
       ELSE '流失' END AS active_status,
  (SELECT category FROM orders o2 WHERE o2.user_id=o.user_id
   GROUP BY category ORDER BY SUM(amount) DESC LIMIT 1) AS top_category
FROM orders o GROUP BY user_id"""

profile=pd.read_sql(q,conn)
print("用户画像表(前10行)：")
print(profile.head(10).to_string())
print(f"\n画像覆盖{len(profile)}个用户")
print(f"\n价值分层分布：\n{profile['value_tier'].value_counts()}")
print(f"\n活跃状态分布：\n{profile['active_status'].value_counts()}")


### 步骤2：进阶练习——RFM分群与交叉分析

2.1 计算RFM三个维度的得分
2.2 画像维度交叉分析

In [ ]:
# 步骤2.1：RFM三维得分
q_rfm = """WITH rfm AS (
  SELECT user_id,
    CAST(JULIANDAY('2026-02-01')-JULIANDAY(MAX(order_date)) AS INTEGER) AS recency,
    COUNT(*) AS frequency,
    SUM(amount) AS monetary
  FROM orders GROUP BY user_id
)
SELECT user_id, recency, frequency, ROUND(monetary,0) AS monetary,
  CASE WHEN recency<=7 THEN 3 WHEN recency<=30 THEN 2 ELSE 1 END AS r,
  CASE WHEN frequency>=10 THEN 3 WHEN frequency>=5 THEN 2 ELSE 1 END AS f,
  CASE WHEN monetary>=1000 THEN 3 WHEN monetary>=300 THEN 2 ELSE 1 END AS m
FROM rfm"""
rfm = pd.read_sql(q_rfm, conn)
rfm['rfm_total'] = rfm['r'] + rfm['f'] + rfm['m']
print("RFM得分分布：")
print(rfm.head(10).to_string())
print(f"\nRFM总分分布：\n{rfm['rfm_total'].value_counts().sort_index()}")
print(f"\n高价值用户(RFM>=7)：{len(rfm[rfm['rfm_total']>=7])}人")

# 步骤2.2：画像交叉分析
print("\n价值分层 x 活跃状态 交叉分析：")
q_cross = """SELECT value_tier, active_status, COUNT(*) AS cnt
FROM (
  SELECT user_id,
    CASE WHEN SUM(amount)>=1000 THEN '高价值'
         WHEN SUM(amount)>=300 THEN '中等' ELSE '低价值' END AS value_tier,
    CASE WHEN JULIANDAY('2026-02-01')-JULIANDAY(MAX(order_date))<=7 THEN '活跃'
         WHEN JULIANDAY('2026-02-01')-JULIANDAY(MAX(order_date))<=30 THEN '沉默'
         ELSE '流失' END AS active_status
  FROM orders GROUP BY user_id
) GROUP BY value_tier, active_status ORDER BY value_tier, active_status"""
cross = pd.read_sql(q_cross, conn)
print(cross.to_string())
print("\n交叉表解读：高价值x流失 = 最值得召回的用户群体")


### 步骤3：挑战练习——多维度画像标签体系

构建包含价值、活跃、品类、消费行为、生命周期五个维度的完整画像表

In [ ]:
# 步骤3：完整用户画像标签体系
q_full = """WITH user_base AS (
  SELECT user_id,
    MIN(order_date) AS first_order,
    MAX(order_date) AS last_order,
    COUNT(*) AS order_count,
    SUM(amount) AS total_spend,
    ROUND(AVG(amount),1) AS avg_order,
    COUNT(DISTINCT category) AS category_diversity,
    ROUND(SUM(CASE WHEN amount>500 THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS big_order_pct,
    CAST(JULIANDAY('2026-02-01')-JULIANDAY(MAX(order_date)) AS INTEGER) AS days_since_last,
    CAST(JULIANDAY(MAX(order_date))-JULIANDAY(MIN(order_date)) AS INTEGER) AS lifecycle_days
  FROM orders GROUP BY user_id
),
with_fav AS (
  SELECT user_id, category AS fav_category FROM (
    SELECT user_id, category,
      ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY SUM(amount) DESC) AS rn
    FROM orders GROUP BY user_id, category
  ) WHERE rn=1
)
SELECT b.*, f.fav_category,
  CASE WHEN b.total_spend>=1000 THEN '高价值'
       WHEN b.total_spend>=300 THEN '中等' ELSE '低价值' END AS value_tier,
  CASE WHEN b.days_since_last<=7 THEN '活跃'
       WHEN b.days_since_last<=30 THEN '沉默' ELSE '流失' END AS active_status,
  CASE WHEN b.lifecycle_days<=7 THEN '新用户'
       WHEN b.lifecycle_days<=90 THEN '成长期'
       ELSE '成熟期' END AS lifecycle_stage
FROM user_base b JOIN with_fav f ON b.user_id=f.user_id"""

full_profile = pd.read_sql(q_full, conn)
print(f"完整用户画像表(前10行，共{len(full_profile.columns)}列)：")
print(full_profile.head(10).to_string())
print(f"\n标签维度覆盖：价值分层、活跃状态、品类偏好、消费行为、生命周期")
print(f"\n品类多样性分布：\n{full_profile['category_diversity'].value_counts().sort_index()}")
print(f"\n各品类偏好用户数：\n{full_profile['fav_category'].value_counts()}")


## 📊 区块4：业务场景应用

### 场景1：拉新活动——筛选高质量新用户

**业务需求**：运营做拉新活动，需要找出"注册30天内消费>500元且购买≥2个品类"的高质量新用户，给他们发专属优惠券。

**分析思路**：
1. 筛选新注册用户(注册30天内)
2. 聚合消费数据
3. 多维度筛选

**SQL实现**：
```sql
WITH new_users AS (
  SELECT user_id FROM users
  WHERE DATEDIFF(CURRENT_DATE(), register_date) <= 30
),
user_stats AS (
  SELECT o.user_id,
    SUM(o.amount) AS total_spend,
    COUNT(DISTINCT o.category) AS category_count,
    COUNT(*) AS order_count
  FROM orders o
  JOIN new_users n ON o.user_id = n.user_id
  GROUP BY o.user_id
)
SELECT * FROM user_stats
WHERE total_spend > 500 AND category_count >= 2
ORDER BY total_spend DESC;
```

**不用画像表的后果**：每次筛选都要重新聚合；多个活动需要用同一批用户时重复计算；无法快速查看用户全貌。

### 场景2：沉默用户激活——精准召回

**业务需求**：找出"曾经高价值但现在沉默(30-90天未购买)"的用户，发召回券。

**SQL实现**：
```sql
SELECT user_id, total_spend, days_since_last, fav_category
FROM user_profile
WHERE active_status = '沉默'
  AND value_tier = '高价值'
  AND lifecycle_stage != '流失'
ORDER BY total_spend DESC
LIMIT 500;
```

**运营策略分级**：
- 沉默30天 + 历史消费1000+ → 发10元券
- 沉默60天 + 历史消费1000+ → 发30元券
- 沉默90天 + 历史消费1000+ → 发50元券 + Push通知
- 沉默30天 + 历史消费300-1000 → 发5元券(测试敏感度)

### 场景3：品类偏好驱动的推荐

**业务需求**：给每个用户推荐其最偏好品类的热销商品(排除已购商品)。

**SQL实现**：
```sql
WITH user_prefs AS (
  SELECT user_id, fav_category FROM user_profile
),
hot_products AS (
  SELECT category, product_id, product_name,
    SUM(sales) AS total_sales,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY SUM(sales) DESC) AS rn
  FROM product_sales
  GROUP BY category, product_id, product_name
),
purchased AS (
  SELECT DISTINCT user_id, product_id FROM orders
)
SELECT up.user_id, hp.product_name
FROM user_prefs up
JOIN hot_products hp ON up.fav_category = hp.category AND hp.rn <= 5
LEFT JOIN purchased p ON up.user_id = p.user_id AND hp.product_id = p.product_id
WHERE p.product_id IS NULL
ORDER BY up.user_id;
```

### 场景4：渠道ROI分析

**业务需求**：对比不同获客渠道的用户质量，为下季度预算分配提供数据支持。

**SQL实现**：
```sql
WITH channel_stats AS (
  SELECT u.channel,
    COUNT(DISTINCT u.user_id) AS users,
    COUNT(DISTINCT o.user_id) * 100.0 / COUNT(DISTINCT u.user_id) AS conversion_rate,
    SUM(COALESCE(o.amount, 0)) * 1.0 / COUNT(DISTINCT u.user_id) AS ltv_per_user,
    AVG(CASE WHEN o.user_id IS NOT NULL THEN o.amount END) AS avg_order_value
  FROM users u
  LEFT JOIN orders o ON u.user_id = o.user_id
  GROUP BY u.channel
)
SELECT *, RANK() OVER (ORDER BY ltv_per_user DESC) AS roi_rank
FROM channel_stats
ORDER BY ltv_per_user DESC;
```

## ✅ 区块5：分级检验清单

### 🟢 初级检验(必须能独立完成的)
- [ ] 能写出用户画像的核心SQL模式(GROUP BY user_id + CASE WHEN)
- [ ] 能计算用户的总消费、订单数、客单价三个基础指标
- [ ] 能根据消费金额做简单的用户分层(高/中/低)
- [ ] 能计算用户的最近活跃天数
- [ ] 能说出画像标签的四种类型

**答案提示**：
- 核心模式：SELECT user_id, SUM(amount), COUNT(*), AVG(amount) FROM orders GROUP BY user_id
- 用户分层：CASE WHEN SUM(amount) >= 1000 THEN '高价值' ...
- 活跃天数：DATEDIFF(CURRENT_DATE(), MAX(order_date))
- 四种类型：数值型(total_spend)、分类型(fav_category)、布尔型(is_vip)、时间型(last_order_date)

### 🟡 中级检验(能解释给同事听的原理)
- [ ] 能解释RFM模型三个维度的业务含义及打分逻辑
- [ ] 能实现品类偏好标签(按金额和按频次两种方式)
- [ ] 能说出画像表设计的五个原则
- [ ] 能计算用户的跨品类数和消费波动率
- [ ] 能实现画像的交叉分析(价值分层 x 活跃状态)

**答案提示**：
- R=最近购买时间(越短越好)、F=购买频次(越多越好)、M=消费金额(越高越好)
- 品类偏好：ROW_NUMBER OVER user_id ORDER BY SUM(amount) DESC，取rn=1
- 五个原则：一行一用户、标签互斥、可增量更新、区分未知和没有、时效和统计分离
- 跨品类数：COUNT(DISTINCT category)；消费波动：STDDEV(amount)
- 交叉分析：GROUP BY value_tier, active_status

### 🔴 高级检验(能举一反三的迁移能力)
- [ ] 能设计完整的用户画像标签体系(20+标签跨5个维度)
- [ ] 能设计画像表的增量更新方案(分区表+每日更新)
- [ ] 能评估画像表对下游运营活动的ROI提升
- [ ] 能处理画像表的性能优化(索引、分区、预聚合)
- [ ] 能将RFM模型扩展为5维度用户评分卡

**答案提示**：
- 标签体系至少覆盖：价值(3)、活跃(2)、品类偏好(3)、消费行为(5)、生命周期(2)五个维度
- 按日期分区，每天增量更新时效标签，全量更新统计标签(每周)
- 做AB实验对比有画像指导的运营活动vs无画像的活动效果
- 对user_id建主键索引，对高频筛选字段建普通索引，按日期分区，冷热分离存储
- 扩展维度：加入社交行为、客服反馈、APP使用时长等非交易数据

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W6D1 复杂SQL拆解 | 画像表是多个CTE链式聚合的结果 |
| **← 前置依赖** | W5D2 窗口函数ROW_NUMBER | 品类偏好排名需要ROW_NUMBER |
| **← 前置依赖** | W5D4 JOIN | 用户表JOIN行为表是画像的基础 |
| **→ 后续用途** | W7 AB实验用户分群 | 实验需要按画像维度分群对比 |
| **→ 后续用途** | W10 精细化运营案例 | 画像标签直接用于运营策略 |
| **业务连接** | RFM模型 | 画像中的价值标签就是RFM的工程实现 |
| **业务连接** | CRM系统 | 画像标签导出到CRM驱动自动化营销 |

### 横向连接：与其他系统的关联

**与推荐系统的连接**：品类偏好标签→推荐系统的用户特征输入；消费层级标签→推荐价格区间。

**与数据仓库的连接**：画像表是典型的数据集市层(DM层)表，从DWD层(明细)→DWS层(汇总)→DM层(画像)的经典数据流。

**与AB实验的连接**：画像标签用于实验前分群(确保实验组和对照组在各个画像维度上均衡)和实验后分析(看不同用户群对实验策略的反应差异)。

## 📝 区块7：当日小结

> **用户画像在SQL里就是GROUP BY user_id + 一堆CASE WHEN多维度聚合。核心原则：一行一用户、标签互斥、可增量更新、区分未知和没有。RFM是价值标签的经典框架，活跃度是时效性最强的标签。品类偏好可以用ROW_NUMBER排名法实现。画像表最终要服务于用户分群和精准营销。**

**三条金句带回家**：
1. 用户画像不是CDP工具的专属——用SQL的GROUP BY + CASE WHEN就能构建80%的画像标签
2. 一行一个用户是画像表不可违背的底线原则
3. 标签要互斥、要能增量更新、要区分"未知"和"没有"

### 🔖 课后拓展

1. **练习**：用SQL实现一个10标签的用户画像表(价值3个+活跃2个+品类2个+消费行为3个)

2. **思考**：如果画像表有1亿用户、50个标签、每天全量更新，需要多少计算资源？如何优化为增量更新？

3. **面试准备**：能画出用户画像表的数据流图(DWD→DWS→DM)，并解释每一层的聚合粒度

4. **预习**：明天要学的活动效果评估，需要用画像标签做活动用户vs非活动用户的对比

---

*03analyst_thinking · Week 6 Day 2 · 2026-05-21*